
# 14 — Analyze Probabilistic Clone Inference Results

**LeukemiPrediag v2.1 — Stage B analysis**

This notebook analyzes the outputs produced by:

1. `12_build_real_features.ipynb`
2. `13_infer_probabilistic_clones.ipynb`

It does **not** train a leukemia-progression classifier and must not be interpreted as clinical validation.

The main goals are to:

- summarize clone-inference results,
- identify the fastest-growing candidate clones,
- inspect posterior uncertainty,
- quantify whether clone-level pooling stabilizes growth estimates,
- inspect high-support co-clonal variant pairs,
- summarize genes represented in multi-variant candidate clones,
- visualize representative longitudinal VAF trajectories,
- export pitch/paper-ready tables and figures.

> Important: a **candidate clone** is a longitudinal co-movement hypothesis inferred from VAF trajectories. It is not proof of phylogenetic ancestry.


In [ ]:

# ============================================================
# 0. Colab / project setup
# ============================================================
from pathlib import Path
import os
import sys
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")

    MYDRIVE = Path("/content/drive/MyDrive")

    # Known project locations first; then a one-level fallback search.
    candidates = [
        MYDRIVE / "leukemia-prediagnose" / "blood-cancer-early-detection",
        MYDRIVE / "blood-cancer-early-detection",
    ]
    candidates += list(MYDRIVE.glob("*/blood-cancer-early-detection"))

    PROJECT_ROOT = next((p for p in candidates if p.exists()), None)

    if PROJECT_ROOT is None:
        raise FileNotFoundError(
            "Could not find blood-cancer-early-detection under /content/drive/MyDrive. "
            "Check that Google Drive is mounted and the project folder exists."
        )
else:
    # Local fallback when the notebook is run from the repository.
    here = Path.cwd()
    if here.name == "notebooks":
        here = here.parent
    PROJECT_ROOT = here

os.chdir(PROJECT_ROOT)

V2_DIR = PROJECT_ROOT / "data" / "processed" / "v2"
CLONE_DIR = V2_DIR / "clone_inference"
SERIAL_PATH = V2_DIR / "real_serial_filtered.csv"

RESULTS_DIR = PROJECT_ROOT / "results" / "clone_analysis_v2"
FIG_DIR = RESULTS_DIR / "figures"
TABLE_DIR = RESULTS_DIR / "tables"
FIG_DIR.mkdir(parents=True, exist_ok=True)
TABLE_DIR.mkdir(parents=True, exist_ok=True)

print("PROJECT_ROOT:", PROJECT_ROOT)
print("CLONE_DIR:", CLONE_DIR)
print("RESULTS_DIR:", RESULTS_DIR)


In [ ]:

# ============================================================
# 1. Verify notebook-13 outputs and load them
# ============================================================
required = {
    "serial": SERIAL_PATH,
    "trajectory": CLONE_DIR / "trajectory_posteriors.csv",
    "pairs": CLONE_DIR / "pair_support.csv",
    "assignments": CLONE_DIR / "clone_assignments.csv",
    "clones": CLONE_DIR / "clone_posteriors.csv",
    "patient_features": CLONE_DIR / "patient_clone_features.csv",
}

for name, path in required.items():
    print(f"{name:18s}", "OK" if path.exists() else "MISSING", path)

missing = [str(p) for p in required.values() if not p.exists()]
if missing:
    raise FileNotFoundError(
        "Notebook 14 needs notebook-13 outputs. Missing:\n" + "\n".join(missing)
    )

serial = pd.read_csv(required["serial"])
trajectory = pd.read_csv(required["trajectory"])
pairs = pd.read_csv(required["pairs"])
assignments = pd.read_csv(required["assignments"])
clones = pd.read_csv(required["clones"])
patient_features = pd.read_csv(required["patient_features"])

sensitivity_path = CLONE_DIR / "threshold_sensitivity.csv"
sensitivity = pd.read_csv(sensitivity_path) if sensitivity_path.exists() else None

audit_path = CLONE_DIR / "clone_inference_audit.json"
audit = json.load(open(audit_path)) if audit_path.exists() else {}

print("\nLoaded:")
for name, df in [
    ("serial", serial),
    ("trajectory", trajectory),
    ("pairs", pairs),
    ("assignments", assignments),
    ("clones", clones),
    ("patient_features", patient_features),
]:
    print(f"{name:18s} shape={df.shape}")

print("\nCohorts:", sorted(serial["cohort"].dropna().unique().tolist()))



## 2. Core audit

This reproduces the key structural counts from notebook 13 and checks that the loaded files are mutually consistent.


In [ ]:

# ============================================================
# 2. Core audit / sanity checks
# ============================================================
n_patients = serial["patient_id"].nunique()
n_variants = trajectory.shape[0]
n_clones = clones.shape[0]
multi = clones[clones["n_variants"] > 1].copy()
n_multi = len(multi)
n_supported_patients = int(patient_features["structure_supported"].fillna(0).astype(int).sum())

core_summary = pd.DataFrame({
    "metric": [
        "patients",
        "patient-variant trajectories",
        "candidate clones",
        "multi-variant candidate clones",
        "patients with supported clone structure",
        "pairwise comparisons evaluated",
        "high-support pairs",
    ],
    "value": [
        n_patients,
        n_variants,
        n_clones,
        n_multi,
        n_supported_patients,
        len(pairs),
        int(pairs["candidate_same_clone"].fillna(False).astype(bool).sum()),
    ],
})

display(core_summary)

if audit:
    print("\nNotebook-13 audit:")
    print(json.dumps(audit, indent=2))

# Basic invariants
assert assignments.shape[0] == trajectory.shape[0], (
    "Every trajectory should have one clone assignment."
)
assert clones["clone_id"].is_unique, "clone_id should be unique in clone_posteriors.csv"


In [ ]:

# Cohort-level summary
cohort_summary = (
    patient_features.groupby("cohort", dropna=False)
    .agg(
        patients=("patient_id", "nunique"),
        patients_with_supported_structure=("structure_supported", "sum"),
        median_n_clones=("n_inferred_clones", "median"),
        median_max_clone_growth=("max_clone_growth", "median"),
        median_max_p_growing=("max_clone_p_growing", "median"),
    )
    .reset_index()
)

display(cohort_summary)
cohort_summary.to_csv(TABLE_DIR / "cohort_summary.csv", index=False)



## 3. Annotate inferred clones with genes and protein changes

Notebook 13 works on `variant_id`. Here we map those variants back to gene/protein annotations from `real_serial_filtered.csv`.


In [ ]:

# ============================================================
# 3. Variant annotation + human-readable clone membership
# ============================================================
variant_ann = (
    serial.sort_values(["patient_id", "variant_id", "time_years"])
    .groupby(["patient_id", "variant_id"], as_index=False)
    .agg(
        gene=("gene", "first"),
        protein_change=("protein_change", "first"),
        mut_key=("mut_key", "first"),
        kind=("kind", "first"),
        first_vaf=("vaf", "first"),
        last_vaf_observed=("vaf", "last"),
    )
)

assignment_ann = assignments.merge(
    variant_ann,
    on=["patient_id", "variant_id"],
    how="left",
    validate="one_to_one",
)

def member_label(row):
    gene = row["gene"] if pd.notna(row["gene"]) else "NA"
    prot = row["protein_change"] if pd.notna(row["protein_change"]) else row["variant_id"]
    return f"{gene}:{prot}"

assignment_ann["member_label"] = assignment_ann.apply(member_label, axis=1)

clone_members = (
    assignment_ann.groupby(["patient_id", "clone_id"], as_index=False)
    .agg(
        genes=("gene", lambda x: ";".join(sorted(set(str(v) for v in x.dropna())))),
        members=("member_label", lambda x: " | ".join(x.astype(str))),
        member_variants=("variant_id", lambda x: " | ".join(x.astype(str))),
    )
)

clone_view = clones.merge(
    clone_members,
    on=["patient_id", "clone_id"],
    how="left",
    validate="one_to_one",
)

multi_view = clone_view[clone_view["n_variants"] > 1].copy()

print("Multi-variant candidate clones:", len(multi_view))
display(
    multi_view.sort_values(
        ["clone_p_growing", "clone_growth_mean"],
        ascending=False
    )[
        [
            "patient_id", "cohort", "clone_id", "n_variants",
            "genes", "members",
            "clone_growth_mean", "clone_growth_ci_low", "clone_growth_ci_high",
            "clone_p_growing", "clone_p_fast_growth",
            "clone_min_pair_support", "clone_mean_pair_support",
        ]
    ].head(20)
)

clone_view.to_csv(TABLE_DIR / "clone_view_annotated.csv", index=False)
multi_view.to_csv(TABLE_DIR / "multivariant_clone_view.csv", index=False)



## 4. Fastest-growing candidate clones

The plot below uses posterior mean growth with the 95% interval produced by notebook 13.

This is **clonal-growth evidence**, not leukemia probability.


In [ ]:

# ============================================================
# 4. Posterior growth intervals for top candidate clones
# ============================================================
TOP_N = min(15, len(multi_view))

if TOP_N == 0:
    print("No multi-variant candidate clones were inferred.")
else:
    top = (
        multi_view.sort_values("clone_growth_mean", ascending=False)
        .head(TOP_N)
        .copy()
        .sort_values("clone_growth_mean")
    )

    labels = [
        f"{p} | {g if g else 'NA'}"
        for p, g in zip(top["patient_id"], top["genes"])
    ]

    y = np.arange(len(top))
    mean = top["clone_growth_mean"].to_numpy(float)
    lo = top["clone_growth_ci_low"].to_numpy(float)
    hi = top["clone_growth_ci_high"].to_numpy(float)

    fig, ax = plt.subplots(figsize=(10, max(5, 0.45 * len(top))))
    ax.errorbar(
        mean,
        y,
        xerr=np.vstack([mean - lo, hi - mean]),
        fmt="o",
        capsize=3,
    )
    ax.axvline(0, linewidth=1)
    ax.set_yticks(y)
    ax.set_yticklabels(labels)
    ax.set_xlabel("Clone growth rate (log-VAF / year)")
    ax.set_title("Top multi-variant candidate clones: posterior growth intervals")
    fig.tight_layout()
    fig.savefig(FIG_DIR / "01_top_clone_growth_intervals.png", dpi=220, bbox_inches="tight")
    plt.show()



## 5. Does clone pooling reduce growth uncertainty?

For each multi-variant candidate clone, compare its posterior standard deviation with the mean posterior SD of its member variants.


In [ ]:

# ============================================================
# 5. Clone-level vs member-variant uncertainty
# ============================================================
member_unc = (
    assignments[["patient_id", "variant_id", "clone_id", "clone_size_variants"]]
    .merge(
        trajectory[["patient_id", "variant_id", "growth_sd"]],
        on=["patient_id", "variant_id"],
        how="left",
        validate="one_to_one",
    )
    .merge(
        clones[["patient_id", "clone_id", "n_variants", "clone_growth_sd"]],
        on=["patient_id", "clone_id"],
        how="left",
        validate="many_to_one",
    )
)

uncertainty = (
    member_unc[member_unc["n_variants"] > 1]
    .groupby(["patient_id", "clone_id"], as_index=False)
    .agg(
        n_variants=("n_variants", "first"),
        clone_sd=("clone_growth_sd", "first"),
        mean_member_sd=("growth_sd", "mean"),
        median_member_sd=("growth_sd", "median"),
    )
)

if len(uncertainty):
    uncertainty["clone_to_member_sd_ratio"] = (
        uncertainty["clone_sd"] / uncertainty["mean_member_sd"]
    )
    uncertainty["lower_clone_uncertainty"] = (
        uncertainty["clone_sd"] < uncertainty["mean_member_sd"]
    )

    print(
        "Median clone/member SD ratio:",
        round(float(uncertainty["clone_to_member_sd_ratio"].median()), 4)
    )
    print(
        "Fraction with lower clone uncertainty:",
        f"{uncertainty['lower_clone_uncertainty'].mean():.1%}"
    )

    display(
        uncertainty.sort_values("clone_to_member_sd_ratio")
    )

    fig, ax = plt.subplots(figsize=(6.5, 6))
    ax.scatter(
        uncertainty["mean_member_sd"],
        uncertainty["clone_sd"],
        s=55,
    )
    lim = max(
        float(uncertainty["mean_member_sd"].max()),
        float(uncertainty["clone_sd"].max()),
    ) * 1.05
    ax.plot([0, lim], [0, lim], linestyle="--", linewidth=1)
    ax.set_xlim(0, lim)
    ax.set_ylim(0, lim)
    ax.set_xlabel("Mean member-variant posterior SD")
    ax.set_ylabel("Clone-level posterior SD")
    ax.set_title("Clone pooling vs individual-variant uncertainty")
    fig.tight_layout()
    fig.savefig(FIG_DIR / "02_clone_vs_variant_uncertainty.png", dpi=220, bbox_inches="tight")
    plt.show()

    uncertainty.to_csv(TABLE_DIR / "clone_uncertainty_comparison.csv", index=False)
else:
    print("No multi-variant clones available for uncertainty comparison.")



## 6. Pairwise clone-support evidence

Notebook 13 combines longitudinal correlation, VAF-ratio stability, and slope similarity into `support`.

The default research threshold is the one stored in the notebook-13 audit (currently expected to be 0.80).


In [ ]:

# ============================================================
# 6. Pair-support distribution and top supported pairs
# ============================================================
default_support = (
    audit.get("config", {}).get("pair_support_threshold", 0.80)
    if audit else 0.80
)

pair_ann_a = variant_ann.rename(columns={
    "variant_id": "variant_a",
    "gene": "gene_a",
    "protein_change": "protein_a",
    "mut_key": "mut_key_a",
})[
    ["patient_id", "variant_a", "gene_a", "protein_a", "mut_key_a"]
]

pair_ann_b = variant_ann.rename(columns={
    "variant_id": "variant_b",
    "gene": "gene_b",
    "protein_change": "protein_b",
    "mut_key": "mut_key_b",
})[
    ["patient_id", "variant_b", "gene_b", "protein_b", "mut_key_b"]
]

pairs_annotated = (
    pairs.merge(pair_ann_a, on=["patient_id", "variant_a"], how="left")
         .merge(pair_ann_b, on=["patient_id", "variant_b"], how="left")
)

supported_pairs = pairs_annotated[
    pairs_annotated["support"] >= float(default_support)
].copy()

print("Default support threshold:", default_support)
print("Pairs evaluated:", len(pairs_annotated))
print("Pairs >= threshold:", len(supported_pairs))

display(
    supported_pairs.sort_values("support", ascending=False)[
        [
            "patient_id", "cohort",
            "gene_a", "protein_a", "gene_b", "protein_b",
            "support", "p_corr", "p_ratio_stable", "p_slope_similar",
            "corr_median", "slope_diff_median",
        ]
    ].head(25)
)

fig, ax = plt.subplots(figsize=(8, 5))
ax.hist(pairs_annotated["support"].dropna(), bins=25)
ax.axvline(float(default_support), linestyle="--", linewidth=1.5)
ax.set_xlabel("Pair support")
ax.set_ylabel("Number of variant pairs")
ax.set_title("Distribution of longitudinal same-clone support")
fig.tight_layout()
fig.savefig(FIG_DIR / "03_pair_support_distribution.png", dpi=220, bbox_inches="tight")
plt.show()

pairs_annotated.to_csv(TABLE_DIR / "pair_support_annotated.csv", index=False)



## 7. Threshold sensitivity

If `threshold_sensitivity.csv` exists, plot how the number of supported structures changes across the precomputed support thresholds.


In [ ]:

# ============================================================
# 7. Threshold sensitivity
# ============================================================
if sensitivity is None:
    print("threshold_sensitivity.csv was not found; skipping this section.")
else:
    display(sensitivity)

    fig, ax = plt.subplots(figsize=(7.5, 5))
    ax.plot(
        sensitivity["pair_support_threshold"],
        sensitivity["multivariant_clones"],
        marker="o",
        label="Multi-variant clones",
    )
    ax.plot(
        sensitivity["pair_support_threshold"],
        sensitivity["patients_with_supported_structure"],
        marker="o",
        label="Patients with supported structure",
    )
    ax.set_xlabel("Pair-support threshold")
    ax.set_ylabel("Count")
    ax.set_title("Clone-structure sensitivity to support threshold")
    ax.legend()
    fig.tight_layout()
    fig.savefig(FIG_DIR / "04_threshold_sensitivity.png", dpi=220, bbox_inches="tight")
    plt.show()



## 8. Genes represented in multi-variant candidate clones

This is a descriptive count, **not an enrichment test**. A formal enrichment analysis would require an explicit background model and careful correction for panel composition and detectability.


In [ ]:

# ============================================================
# 8. Gene composition of multi-variant candidate clones
# ============================================================
multi_members = assignment_ann[
    assignment_ann["clone_size_variants"] > 1
].copy()

if len(multi_members):
    gene_counts = (
        multi_members.dropna(subset=["gene"])
        .groupby("gene")
        .agg(
            variants_in_multivariant_clones=("variant_id", "count"),
            patients=("patient_id", "nunique"),
            candidate_clones=("clone_id", "nunique"),
        )
        .sort_values("variants_in_multivariant_clones", ascending=False)
        .reset_index()
    )

    display(gene_counts.head(25))
    gene_counts.to_csv(TABLE_DIR / "genes_in_multivariant_clones.csv", index=False)

    top_genes = gene_counts.head(15).sort_values("variants_in_multivariant_clones")
    fig, ax = plt.subplots(figsize=(8, max(4.5, 0.35 * len(top_genes))))
    ax.barh(
        top_genes["gene"],
        top_genes["variants_in_multivariant_clones"],
    )
    ax.set_xlabel("Variants assigned to multi-variant candidate clones")
    ax.set_ylabel("Gene")
    ax.set_title("Gene composition of supported multi-variant clones")
    fig.tight_layout()
    fig.savefig(FIG_DIR / "05_gene_composition_multivariant_clones.png", dpi=220, bbox_inches="tight")
    plt.show()
else:
    print("No multi-variant clone members found.")



## 9. Example longitudinal VAF trajectories

Automatically choose a multi-variant candidate clone with strong evidence and plot its member trajectories.

This is useful for manual scientific inspection and for explaining the method in a pitch deck or paper.


In [ ]:

# ============================================================
# 9. Plot one representative multi-variant candidate clone
# ============================================================
if len(multi_view):
    example = (
        multi_view.assign(
            support_for_sort=multi_view["clone_min_pair_support"].fillna(0)
        )
        .sort_values(
            ["clone_p_growing", "support_for_sort", "n_variants"],
            ascending=False,
        )
        .iloc[0]
    )

    example_patient = example["patient_id"]
    example_clone = example["clone_id"]

    member_ids = assignments.loc[
        (assignments["patient_id"] == example_patient) &
        (assignments["clone_id"] == example_clone),
        "variant_id"
    ].tolist()

    ex = serial[
        (serial["patient_id"] == example_patient) &
        (serial["variant_id"].isin(member_ids))
    ].copy()

    print("Example patient:", example_patient)
    print("Example clone:", example_clone)
    print("Members:", example["members"])
    print("Posterior growth mean:", example["clone_growth_mean"])
    print("P(growing):", example["clone_p_growing"])
    print("Min pair support:", example["clone_min_pair_support"])

    fig, ax = plt.subplots(figsize=(9, 5.5))
    for variant_id, g in ex.groupby("variant_id"):
        gene = g["gene"].dropna().iloc[0] if g["gene"].notna().any() else "NA"
        prot = (
            g["protein_change"].dropna().iloc[0]
            if g["protein_change"].notna().any()
            else variant_id
        )
        g = g.sort_values("time_years")
        ax.plot(
            g["time_years"],
            g["vaf"],
            marker="o",
            label=f"{gene}:{prot}",
        )

    ax.set_xlabel("Time (years)")
    ax.set_ylabel("VAF")
    ax.set_title(f"Representative candidate clone — {example_patient}")
    ax.legend()
    fig.tight_layout()
    fig.savefig(FIG_DIR / "06_example_multivariant_clone_trajectory.png", dpi=220, bbox_inches="tight")
    plt.show()
else:
    print("No multi-variant candidate clone is available for example plotting.")



## 10. Patient-level clone landscape

Summarize how many patients have supported multi-variant structure and how clone-growth evidence is distributed.


In [ ]:

# ============================================================
# 10. Patient-level landscape
# ============================================================
fig, ax = plt.subplots(figsize=(8, 5))
ax.hist(patient_features["max_clone_growth"].dropna(), bins=25)
ax.axvline(0, linewidth=1)
ax.set_xlabel("Maximum inferred clone growth per patient")
ax.set_ylabel("Patients")
ax.set_title("Patient-level maximum clone growth")
fig.tight_layout()
fig.savefig(FIG_DIR / "07_patient_max_clone_growth.png", dpi=220, bbox_inches="tight")
plt.show()

support_by_cohort = (
    patient_features.groupby("cohort", as_index=False)
    .agg(
        patients=("patient_id", "nunique"),
        supported=("structure_supported", "sum"),
    )
)
support_by_cohort["supported_fraction"] = (
    support_by_cohort["supported"] / support_by_cohort["patients"]
)

display(support_by_cohort)
support_by_cohort.to_csv(TABLE_DIR / "supported_structure_by_cohort.csv", index=False)



## 11. Export a compact analysis summary

This JSON is suitable as the machine-readable summary for later pitch-deck or manuscript figure generation.


In [ ]:

# ============================================================
# 11. Export analysis summary
# ============================================================
summary = {
    "research_use_only": True,
    "not_a_leukemia_risk_model": True,
    "n_patients": int(n_patients),
    "n_patient_variant_trajectories": int(n_variants),
    "n_pairwise_comparisons": int(len(pairs)),
    "n_high_support_pairs": int(
        pairs["candidate_same_clone"].fillna(False).astype(bool).sum()
    ),
    "n_candidate_clones": int(n_clones),
    "n_multivariant_candidate_clones": int(n_multi),
    "n_patients_with_supported_structure": int(n_supported_patients),
    "default_pair_support_threshold": float(default_support),
}

if len(uncertainty):
    summary.update({
        "median_clone_to_member_sd_ratio": float(
            uncertainty["clone_to_member_sd_ratio"].median()
        ),
        "fraction_multivariant_clones_with_lower_uncertainty": float(
            uncertainty["lower_clone_uncertainty"].mean()
        ),
    })

summary_path = RESULTS_DIR / "clone_analysis_summary.json"
with open(summary_path, "w") as f:
    json.dump(summary, f, indent=2)

core_summary.to_csv(TABLE_DIR / "core_summary.csv", index=False)

print(json.dumps(summary, indent=2))
print("\nSaved summary:", summary_path)
print("Figures:", FIG_DIR)
print("Tables:", TABLE_DIR)



## Interpretation boundary

What this notebook can support:

- longitudinal clone-growth measurement,
- posterior uncertainty for variant and candidate-clone growth,
- candidate co-clonal structure based on VAF co-movement,
- comparison of clone-level vs independent-variant uncertainty,
- descriptive gene/cohort summaries.

What it **cannot** support with the current cohorts:

- probability that a patient will develop leukemia,
- clinical sensitivity/specificity for future leukemia,
- proof that two variants are in the same biological cell lineage,
- causal interpretation of genes enriched among candidate clones.

The next clinical modeling stage requires an outcome-linked longitudinal cohort.
